1. Requirement

Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "id string, first_name string, last_name string, address string, skills string, contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
    .option("header","true")
    .option("quote","\"")
    .option("escape","\"")
    .schema(offline_students_schema)
    .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/students_offline.csv")
)

# overwriteSchema allows the schema of the existing table to be overwritten
offline_students_raw_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev_catalog.spark_db.offline_students_raw")

2. Requirement\
Prepare an offline_students table which is ready for analysis.

In [0]:
from pyspark.sql.functions import parse_json
offline_students_raw_df_2 = spark.read.table("dev_catalog.spark_db.offline_students_raw")
offline_students_df = (
    offline_students_raw_df_2.withColumns({
        "address" : parse_json("address"),
        "skills" : parse_json("skills"),
        "contacts" : parse_json("contacts")
    })
)
# offline_students_df.display()
offline_students_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev_catalog.spark_db.offline_var_students")
offline_students_df.display()


####3. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

2.1 What is country wise student count.

In [0]:
%sql
-- Variant object element names are case sensitive

select cast(address:Country as string), count(*) as count 
from dev_catalog.spark_db.offline_var_students
group by cast(address:Country as string)

In [0]:
%sql
with offline_students_skills(
  select id, first_name, last_name,  cast(value:Skill as string), cast
  (value:YearsOfExperience as int)
  from dev_catalog.spark_db.offline_var_students,
  lateral variant_explode_outer(skills) 
) 
select * from offline_students_skills
where skill like '%Spark%' and yearsofexperience > 1





2.3 Find all students who did not provide phone or whatsapp.

In [0]:
%sql
select id, first_name, last_name, contacts:email 
from dev_catalog.spark_db.offline_var_students
where contacts:phone is null and contacts:whatsapp is null